# OpenUBA SDK: Visualizations

This notebook demonstrates creating visualizations and rendering with multiple backends.

**SDK functions covered:** `create_visualization`, `publish_visualization`, `list_visualizations`, `VisualizationContext`

In [ ]:
import openuba
print(f'OpenUBA SDK v{openuba.__version__}')

## 1. Create Visualizations with Different Backends

In [ ]:
viz_ids = []

backends = [
    ('matplotlib', 'Login Anomaly Heatmap'),
    ('seaborn', 'Risk Score Distribution'),
    ('plotly', 'Interactive Timeline'),
    ('bokeh', 'Real-time Monitor'),
    ('altair', 'Entity Behavior Chart'),
    ('networkx', 'Lateral Movement Graph'),
]

for backend, name in backends:
    try:
        viz = openuba.create_visualization(
            name=f'nb-{name.lower().replace(" ", "-")}',
            backend=backend,
            description=f'{name} using {backend}',
        )
        viz_id = viz.get('id')
        viz_ids.append(viz_id)
        print(f'Created ({backend}): {viz.get("name")} -> output_type={viz.get("output_type")}')
    except Exception as e:
        print(f'create_visualization ({backend}): {e}')

## 2. List All Visualizations

In [ ]:
try:
    all_viz = openuba.list_visualizations()
    print(f'Total visualizations: {len(all_viz)}')
    for v in all_viz:
        print(f'  - {v.get("name")}: backend={v.get("backend")}, status={v.get("status")}')
except Exception as e:
    print(f'list_visualizations: {e}')

## 3. Publish a Visualization

In [ ]:
if viz_ids:
    try:
        pub = openuba.publish_visualization(viz_ids[0])
        print(f'Published: {pub}')
    except Exception as e:
        print(f'publish_visualization: {e}')
else:
    print('Skipped: no viz IDs')

## 4. VisualizationContext: Render with Matplotlib

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Create a matplotlib figure
fig, ax = plt.subplots(figsize=(8, 4))
np.random.seed(42)
risk_scores = np.random.beta(2, 5, 1000)
ax.hist(risk_scores, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_title('Entity Risk Score Distribution')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Count')
ax.axvline(x=0.7, color='red', linestyle='--', label='Threshold')
ax.legend()

# Render using VisualizationContext
svg_output = openuba.VisualizationContext.render(fig)
print(f'Matplotlib SVG rendered: {len(svg_output)} chars')
print(f'Starts with: {svg_output[:80]}...')
plt.close(fig)

## 5. VisualizationContext: Render with Seaborn

In [ ]:
import seaborn as sns
import pandas as pd

# Create sample data
df = pd.DataFrame({
    'hour': list(range(24)) * 7,
    'day': [d for d in ['Mon','Tue','Wed','Thu','Fri','Sat','Sun'] for _ in range(24)],
    'logins': np.random.poisson(10, 168),
})
pivot = df.pivot_table(values='logins', index='day', columns='hour', aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 4))
sns_plot = sns.heatmap(pivot, cmap='YlOrRd', ax=ax)
ax.set_title('Login Activity Heatmap')

svg_output = openuba.VisualizationContext.render(fig, backend='seaborn')
print(f'Seaborn SVG rendered: {len(svg_output)} chars')
plt.close(fig)

## 6. VisualizationContext: Render with Plotly

In [ ]:
import plotly.graph_objects as go

# Create plotly figure
fig_plotly = go.Figure()
fig_plotly.add_trace(go.Scatter(
    x=list(range(100)),
    y=np.cumsum(np.random.randn(100)),
    mode='lines',
    name='Risk Score Trend',
))
fig_plotly.update_layout(title='Entity Risk Over Time', xaxis_title='Time', yaxis_title='Risk')

json_output = openuba.VisualizationContext.render(fig_plotly)
print(f'Plotly JSON rendered: {len(json_output)} chars')
print(f'Contains data: {"Risk Score Trend" in json_output}')

## 7. VisualizationContext: Render with NetworkX

In [ ]:
import networkx as nx

# Create a lateral movement graph
G = nx.DiGraph()
G.add_edges_from([
    ('workstation-1', 'dc-01'),
    ('workstation-1', 'fileserver'),
    ('dc-01', 'dc-02'),
    ('dc-02', 'db-server'),
    ('fileserver', 'backup-server'),
    ('workstation-2', 'dc-01'),
])

svg_output = openuba.VisualizationContext.render(G, backend='networkx')
print(f'NetworkX SVG rendered: {len(svg_output)} chars')
print(f'Graph nodes: {list(G.nodes())}')
plt.close('all')

## 8. VisualizationContext: Backend Output Map

In [ ]:
print('Supported backends and output formats:')
for backend, output in openuba.VisualizationContext.BACKEND_OUTPUT_MAP.items():
    print(f'  {backend:15s} -> {output}')

In [ ]:
print('\n=== VISUALIZATIONS COMPLETE ===')
print(f'Visualizations created via API: {len(viz_ids)}')
print('Backends rendered: matplotlib, seaborn, plotly, networkx')
print('Functions tested: create_visualization, publish_visualization,')
print('  list_visualizations, VisualizationContext.render()')
print('All visualization checks passed!')